In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

# Load the development data
dev_data = pd.read_csv('Dev_data_to_be_shared.csv')

# Check basic info and null values
print("Initial Data Overview:")
print(dev_data.info())
print("\nMissing values:")
print(dev_data.isnull().sum())

Initial Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96806 entries, 0 to 96805
Columns: 1216 entries, account_number to onus_attribute_48
dtypes: float64(1189), int64(27)
memory usage: 898.1 MB
None

Missing values:
account_number                 0
bad_flag                       0
onus_attribute_1           25231
transaction_attribute_1    25231
transaction_attribute_2    25231
                           ...  
onus_attribute_44          85196
onus_attribute_45          85196
onus_attribute_46          85196
onus_attribute_47          85196
onus_attribute_48          85196
Length: 1216, dtype: int64


In [ ]:
dev_data.fillna(dev_data.mean(),inplace=True)
corr_matrix = dev_data.corr().abs()
upper = corr_matrix.where(np.triu(pd.np.ones(corr_matrix.shape), k=1).astype(bool))
# Drop features with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
dev_data.drop(columns=to_drop, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = dev_data.drop(columns=['bad_flag'])
y = dev_data['bad_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
print("AUC-ROC:", roc_auc_score(y_test, y_pred))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(eval_metric='auc')
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
print("AUC-ROC:", roc_auc_score(y_test, y_pred_xgb))

In [ ]:
print("Logistic Regression Report:")
print(classification_report(y_test, model.predict(X_test)))

print("XGBoost Report:")
print(classification_report(y_test, xgb_model.predict(X_test)))

In [ ]:
validation_data_cleaned = validation_data.drop(columns=['account_number'])
predictions = xgb_model.predict_proba(validation_data_cleaned)[:, 1]

In [ ]:
output = pd.DataFrame({'account_number': account_numbers, 'predicted_probability': predictions})
output.to_csv('validation_predictions.csv', index=False)